# RAG-Based Offline PDF Assistant — walkthrough

All the logic lives in `pdf_qa_core.py`. This notebook imports it and runs the
pipeline one stage at a time, so each step of the RAG flow can be inspected on
its own.

**Pipeline**

`PDF → text + tables + OCR → chunks → embeddings → FAISS → retrieval → Mistral (Ollama) → answer`

**Before running:** activate the virtual environment, make sure Ollama is
running, and place a PDF named `sample.pdf` next to this notebook.
For the desktop interface instead, run `python gui_app.py`.

## 1. Environment check

Confirms the kernel, OCR and the local LLM are all reachable before anything else runs.

In [ ]:
import sys, requests
print("Python:", sys.version.split()[0])
print("Kernel:", sys.executable)          # should end in venv\Scripts\python.exe

import pytesseract
print("Tesseract:", pytesseract.get_tesseract_version())

try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()
    print("Ollama models:", [m["name"] for m in tags.get("models", [])])
except Exception as exc:
    print("Ollama not reachable:", exc)

## 2. Import the core module

`autoreload` picks up edits to `pdf_qa_core.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import pdf_qa_core as core

PDF_PATH = "./sample.pdf"

## 3. Extraction

Two libraries, each used for what it does best:

- **pdfplumber** reads text in layout order and detects tables.
- **PyMuPDF** pulls the bytes of embedded images.

A scanned page has no text layer, so it appears here as an image rather than text.

In [ ]:
text_blocks, tables, images = core.extract_pdf_content(PDF_PATH)

print(f"pages with a text layer : {len(text_blocks)}")
print(f"tables detected         : {len(tables)}")
print(f"embedded images         : {len(images)}")

if text_blocks:
    print("\n--- first 300 characters of page", text_blocks[0]["page"], "---")
    print(text_blocks[0]["text"][:300])

## 4. OCR

Each image is cleaned before Tesseract sees it: grayscale, doubled contrast,
2× upscale, then sharpen. Tesseract is sensitive to resolution and contrast,
so this preprocessing is what makes text on noisy scans readable.

In [ ]:
ocr_blocks = core.ocr_images(images)
print(f"images that produced text: {len(ocr_blocks)}")

if ocr_blocks:
    print("\n" + ocr_blocks[0]["text"][:400])

## 5. Tables

`extract_tables()` returns rows and columns, which are flattened into text so
they can be embedded like any other block. Without this step the table data is
extracted and then discarded, and table questions cannot be answered.

In [ ]:
table_blocks = core.tables_to_blocks(tables)
print(f"table blocks: {len(table_blocks)}")

if table_blocks:
    print("\n" + table_blocks[0]["text"][:500])

## 6. Chunking

The embedding model reads only about the first 256 tokens of any input, so one
page per chunk leaves most of a dense page unsearchable. Text is split into
200-word windows with 40 words of overlap, so a sentence spanning a boundary
still sits whole inside one chunk.

In [ ]:
all_blocks = core.chunk_blocks(text_blocks + ocr_blocks + table_blocks)

print(f"blocks before chunking: {len(text_blocks) + len(ocr_blocks) + len(table_blocks)}")
print(f"chunks after chunking : {len(all_blocks)}")

lengths = [len(c["text"].split()) for c in all_blocks]
if lengths:
    print(f"words per chunk: min {min(lengths)}, max {max(lengths)}")

## 7. Embeddings and the FAISS index

`all-MiniLM-L6-v2` maps each chunk to 384 numbers positioned so that similar
meanings land close together. `IndexFlatL2` is exact brute-force search, which
is the right choice at this scale. The model's output vectors are normalised,
so ranking by L2 distance matches ranking by cosine similarity.

In [ ]:
index = core.PDFIndex()
index.add_documents(all_blocks)

print("vectors indexed :", index.index.ntotal)
print("vector dimension:", index.index.d)

## 8. Retrieval on its own

Worth running separately: it shows what the model will actually see. If an
answer is wrong, this is the first place to look, because a retrieval miss and
a generation mistake need different fixes.

In [ ]:
question = "What is this document about?"

context = index.retrieve_context(question, top_k=4)
print(context[:1200])

## 9. Generation

The retrieved context and the question are sent to Mistral running locally through Ollama.

In [ ]:
answer = core.query_ollama_http(question, context)
print(answer)

## 10. The whole pipeline in one call

`run_pipeline` does steps 3 to 7 together and reports what it indexed.

In [ ]:
index, stats = core.run_pipeline(PDF_PATH)
stats

## 11. Interactive Q&A

Asks questions in a loop, keeps the last few turns as conversation context, and
writes every exchange to `logs/`.

In VS Code the input box appears at the **top of the window**, not in the
notebook. Type `exit` to stop.

In [ ]:
core.run_qa_interface(PDF_PATH)

## 12. Summary

Retrieves the chunks closest to a summary query and asks the model to condense them.

In [ ]:
# core.summarize_pdf(PDF_PATH)

## Known limitations

- **Not truly multimodal.** Images are converted to text by OCR; a chart with no
  readable labels contributes nothing. A vision model would be needed for that.
- **The index is rebuilt on every run.** `faiss.write_index` plus the chunk texts
  would make it persistent.
- **Retrieval scores are ignored.** A relevance threshold would let the assistant
  say "not in this document" before calling the model.
- **No reranking.** A cross-encoder over the top 20 candidates would improve
  precision on harder questions.